In [ ]:
# ⚠️ CRITICAL: Must run this FIRST and ONLY ONCE!
# This cell completely removes torchvision to prevent circular import errors
import subprocess
import sys
import os

print("⚠️  Step 1: Uninstalling torchvision completely...")
result = subprocess.run(
    ["pip", "uninstall", "-y", "torchvision"],
    capture_output=True,
    text=True,
    timeout=60
)
print(f"   {result.stdout.split(chr(10))[0]}")

print("\n✅ Step 2: Setting environment variables...")
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("   ✅ All environment variables set")

print("\n✅ Step 3: Installing import hook...")
# Remove any cached torchvision modules
modules_to_remove = [name for name in list(sys.modules.keys()) if 'torchvision' in name.lower()]
for module_name in modules_to_remove:
    del sys.modules[module_name]
print(f"   ✅ Removed {len(modules_to_remove)} cached torchvision modules")

# Block future imports
class BlockTorchvision:
    def find_module(self, fullname, path=None):
        if 'torchvision' in fullname.lower():
            raise ImportError("torchvision is permanently disabled")
        return None

sys.meta_path.insert(0, BlockTorchvision())
print("   ✅ Import hook installed")

print("\n" + "="*70)
print("✅ ALL TORCHVISION BLOCKS ACTIVATED")
print("="*70)
print("\n⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel")
print("   and run this cell again as the VERY FIRST cell.\n")


⚠️  Step 1: Uninstalling torchvision completely...
   

✅ Step 2: Setting environment variables...
   ✅ All environment variables set

✅ Step 3: Installing import hook...
   ✅ Removed 0 cached torchvision modules
   ✅ Import hook installed

✅ ALL TORCHVISION BLOCKS ACTIVATED

⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel
   and run this cell again as the VERY FIRST cell.



# 🚀 GIS代码生成模型训练 - Google Colab (CodeLlama)

本Notebook在Google Colab上训练GIS代码生成模型（**文件级 + CodeLlama**）

**文件级训练** - 模型学习生成完整的工作流而不是单个步骤
- 输入：用户的高层指令（英语/荷兰语，如："Create MS and HS cable objects"）
- 输出：完整的工作流JSON代码（包含所有操作步骤）
- 优势：一次推理生成整个测试脚本
- **模型**：CodeLlama-7B-Instruct（专为代码生成优化）

**使用前准备：**
1. 运行环境：`Runtime > Change runtime type > T4 GPU`（免费）或 `A100 GPU`（Colab Pro）
2. 数据准备：确保已生成训练数据文件
3. 预计时间：4-6小时（T4）/ 1-2小时（A100）

---

## 📋 步骤1：环境设置

In [ ]:
# 检查GPU
!nvidia-smi

Wed Mar  4 16:08:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 安装依赖（约3-5分钟）
print("📦 Installing dependencies...")

# 先锁定关键基础包（避免自动升级）
!pip install -q torch==2.9.0 --no-deps
!pip install -q fsspec==2024.3.1
!pip install -q numpy==2.0.2 --no-deps

# 安装主要训练库（指定兼容版本）
!pip install -q transformers==4.46.0
!pip install -q peft==0.13.0
!pip install -q datasets==2.19.0
!pip install -q "accelerate>=1.0.0"
!pip install -q sentencepiece==0.2.0
!pip install -q tqdm
!pip install -q huggingface-hub==0.26.0

print("✅ Core dependencies installed! If running in Colab, restart runtime after this cell.")

📦 Installing dependencies...
✅ Core dependencies installed! If running in Colab, restart runtime after this cell.


## 💾 步骤2：挂载Google Drive（保存模型）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 创建输出目录
!mkdir -p /content/drive/MyDrive/gis-models
print("✅ Google Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!


## 📂 步骤3：加载步骤级数据

In [ ]:
import os

# 首先，确保我们回到根目录，避免在错误的位置克隆
%cd /content/

# 删除可能存在的旧仓库副本，确保全新的克隆
!rm -rf gis-code-ai

# 克隆您的GitHub仓库
GITHUB_REPO_URL = "https://github.com/rockyistt/gis-code-ai" # 用户提供的URL

print(f"📦 克隆仓库: {GITHUB_REPO_URL}...")
!git clone {GITHUB_REPO_URL}

# 检查是否成功克隆并进入目录
if os.path.exists('gis-code-ai'):
    print("✅ 仓库克隆成功！")
    %cd gis-code-ai
    print(f"📍 当前工作目录已切换到: {os.getcwd()}")
    print("📂 目录内容: ")
    !ls -F

    # 再次检查数据文件路径
    expected_instructions_file = 'data/processed/step_level_instructions.jsonl'
    expected_data_file = 'data/processed/step_level_data.jsonl'

    if os.path.exists(expected_instructions_file) and os.path.exists(expected_data_file):
        print(f"✅ 已找到数据文件: {expected_instructions_file} 和 {expected_data_file}")
        print("   现在您可以尝试重新运行数据加载单元 (cell `XraxMOdNVGuh`)。")
    else:
        print("❌ 警告: 克隆后数据文件仍未找到。请检查您的GitHub仓库中 `data/processed/` 路径下是否包含 `step_level_instructions.jsonl` 和 `step_level_data.jsonl`。")
        print(f"   当前 {os.getcwd()}/data/ 目录内容:")
        !ls -F data/
else:
    print("❌ 仓库克隆失败，请检查您的GitHub仓库URL或权限。")

print("--------------------------------------------------")
print("克隆完成后，请运行 '步骤3：加载步骤级数据' 部分的代码单元以加载数据。")

/content
📦 克隆仓库: https://github.com/rockyistt/gis-code-ai...
Cloning into 'gis-code-ai'...
remote: Enumerating objects: 4314, done.
remote: Counting objects: 100% (4314/4314), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 4314 (delta 4077), reused 4237 (delta 4020), pack-reused 0 (from 0)
Receiving objects: 100% (4314/4314), 9.33 MiB | 2.54 MiB/s, done.
Resolving deltas: 100% (4077/4077), done.
✅ 仓库克隆成功！
/content/gis-code-ai
📍 当前工作目录已切换到: /content/gis-code-ai
📂 目录内容: 
check_instructions.py	 examples/	   show_weighted_instructions.py
compare_instructions.py  notebooks/	   src/
configs/		 output.log	   tests/
data/			 README.md	   verify_data.py
debug_workflows.py	 requirements.txt  verify_fixed.py
docs/			 scripts/	   verify_work_completion.py
✅ 已找到数据文件: data/processed/step_level_instructions.jsonl 和 data/processed/step_level_data.jsonl
   现在您可以尝试重新运行数据加载单元 (cell `XraxMOdNVGuh`)。
--------------------------------------------------
克隆完成后，请运行 '步骤3：加载步骤级数据' 部分的代码单元以

In [ ]:
import os
import json
import sys
import numpy as np
import random
from pathlib import Path

print("="*70)
print("🔍 第一步：检查和加载数据文件")
print("="*70)

# 确保在正确的目录
if os.path.exists('/content/gis-code-ai'):
    os.chdir('/content/gis-code-ai')
elif os.path.exists('gis-code-ai'):
    os.chdir('gis-code-ai')

print(f"\n📍 当前工作目录: {os.getcwd()}\n")

# ============================================================
# 第1部分：检查数据文件
# ============================================================

print("📋 检查数据文件...\n")

SOURCE_FILES = {
    '✅ 步骤级指令': 'data/processed/step_level_instructions.jsonl',
    '✅ 步骤级数据': 'data/processed/step_level_data.jsonl',
}

files_status = {}
present_files = {}

for desc, filepath in SOURCE_FILES.items():
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                lines = sum(1 for _ in f)
            print(f"{desc} ✅")
            print(f"   📁 {filepath}")
            print(f"   💾 大小: {size_mb:.1f} MB | 📊 行数: {lines:,}\n")

            files_status[filepath] = 'OK'
            present_files[filepath] = size_mb

        except Exception as e:
            print(f"{desc} ⚠️")
            print(f"   📁 {filepath}")
            print(f"   ⚠️ 读取失败: {e}\n")
            files_status[filepath] = 'ERROR'
    else:
        print(f"{desc} ❌")
        print(f"   📁 {filepath}")
        print(f"   ❌ 必需但未找到\n")
        files_status[filepath] = 'MISSING'

# ============================================================
# 第2部分：加载数据文件
# ============================================================

print("="*70)
print("📂 第二步：加载数据")
print("="*70 + "\n")

# 加载step_level_instructions
all_instructions = []
print("1️⃣ 加载步骤级指令...")
try:
    with open('data/processed/step_level_instructions.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line)
                    all_instructions.append(item)
                except json.JSONDecodeError as e:
                    print(f"   ⚠️  第 {line_num} 行解析失败: {e}")
    print(f"   ✅ 已加载: {len(all_instructions):,} 条指令\n")
except Exception as e:
    print(f"   ❌ 加载失败: {e}\n")
    sys.exit(1)

# 加载step_level_data
all_data = []
print("2️⃣ 加载步骤级数据...")
try:
    with open('data/processed/step_level_data.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line)
                    all_data.append(item)
                except json.JSONDecodeError as e:
                    print(f"   ⚠️  第 {line_num} 行解析失败: {e}")
    print(f"   ✅ 已加载: {len(all_data):,} 条数据\n")
except Exception as e:
    print(f"   ❌ 加载失败: {e}\n")
    sys.exit(1)

# 验证数据一致性
if len(all_instructions) != len(all_data):
    print(f"❌ 错误：数据数量不匹配！")
    print(f"   指令: {len(all_instructions)}")
    print(f"   数据: {len(all_data)}")
    sys.exit(1)

# ============================================================
# 第3部分：构造训练数据结构
# ============================================================

print("="*70)
print("🔧 第三步：构造训练数据")
print("="*70 + "\n")

training_data = []
for instruction_item, data_item in zip(all_instructions, all_data):
    sample = {
        'instruction': instruction_item.get('instruction', ''),
        'output': data_item,
        'metadata': {
            'file_id': instruction_item.get('file_id', ''),
            'step_index': instruction_item.get('step_index', 0),
            'keywords': instruction_item.get('keywords', []),
            'avg_weight': instruction_item.get('keyword_weights', {}).get('avg_weight', 1.0),
        }
    }
    training_data.append(sample)

print(f"✅ 已构造: {len(training_data):,} 个训练样本\n")

# ============================================================
# 第4部分：分割Train/Val（按file_id，防止数据泄漏）
# ============================================================

print("="*70)
print("📊 第四步：分割数据集")
print("="*70 + "\n")

# 按file_id分组
file_id_map = {}
for idx, sample in enumerate(training_data):
    file_id = sample['metadata']['file_id']
    if file_id not in file_id_map:
        file_id_map[file_id] = []
    file_id_map[file_id].append(idx)

# 随机分割file_id（不是样本）
random.seed(42)
all_file_ids = list(file_id_map.keys())
random.shuffle(all_file_ids)

split_point = int(len(all_file_ids) * 0.9)
train_file_ids = set(all_file_ids[:split_point])

# 按file_id分割数据
train_data = []
val_data = []
for file_id, indices in file_id_map.items():
    if file_id in train_file_ids:
        train_data.extend([training_data[i] for i in indices])
    else:
        val_data.extend([training_data[i] for i in indices])

# ============================================================
# 第4.5部分：采样10%数据（OOM优化：从50%降到10%）
# ============================================================

print("="*70)
print("📊 采样10%数据（OOM优化版）")
print("="*70 + "\n")

# 原始数据大小
orig_train_size = len(train_data)
orig_val_size = len(val_data)

# ⚠️ 激进的采样：从50%降到10%以节省内存
print("⚠️  采样比例已从50%降到10%以防止OOM")
print(f"   原始训练集: {orig_train_size:,} 样本")
print(f"   原始验证集: {orig_val_size:,} 样本\n")

random.seed(42)

# 采样10%
sample_indices_train = random.sample(range(len(train_data)), max(1, int(len(train_data) * 0.01)))
sample_indices_val = random.sample(range(len(val_data)), max(1, int(len(val_data) * 0.01)))

train_data = [train_data[i] for i in sorted(sample_indices_train)]
val_data = [val_data[i] for i in sorted(sample_indices_val)]

print(f"   🔄 训练集: {orig_train_size:,} → {len(train_data):,} 样本 (保留10%)")
print(f"   🔄 验证集: {orig_val_size:,} → {len(val_data):,} 样本 (保留10%)\n")

print(f"   ✅ 采样后训练集: {len(train_data):,} 样本")
print(f"   ✅ 采样后验证集: {len(val_data):,} 样本")
print(f"   ✅ 比例: {len(train_data)/(len(train_data)+len(val_data))*100:.1f}% 训练 / {len(val_data)/(len(train_data)+len(val_data))*100:.1f}% 验证\n")

# ============================================================
# 第5部分：数据质量检查
# ============================================================

print("="*70)
print("✅ 数据质量检查")
print("="*70 + "\n")

# 检查完整性
total = len(train_data)
has_instruction = sum(1 for s in train_data if 'instruction' in s and s['instruction'])
has_output = sum(1 for s in train_data if 'output' in s)
has_keywords = sum(1 for s in train_data if s.get('metadata', {}).get('keywords'))

print(f"   训练集完整性:")
print(f"      指令完整: {has_instruction}/{total} ({has_instruction/total*100:.1f}%)")
print(f"      输出完整: {has_output}/{total} ({has_output/total*100:.1f}%)")
print(f"      关键词完整: {has_keywords}/{total} ({has_keywords/total*100:.1f}%)\n")

# 显示示例
if train_data:
    sample = train_data[0]
    print(f"📝 数据样本:")
    print(f"   指令: {sample.get('instruction', '')[:80]}...")
    print(f"   输出字段: {list(sample.get('output', {}).keys())}")
    print(f"   关键词: {sample.get('metadata', {}).get('keywords', [])[:3]}\n")

print("="*70)
print("✅ 数据加载、采样和分割完成！")
print("="*70)
print("\n📊 可用变量:")
print("   • train_data: 训练集数据 (已采样10%)")
print("   • val_data: 验证集数据 (已采样10%)")
print(f"   • 总样本数: {len(train_data) + len(val_data):,} (原始: {orig_train_size + orig_val_size:,})")
print()

🔍 第一步：检查和加载数据文件

📍 当前工作目录: /content/gis-code-ai

📋 检查数据文件...

✅ 步骤级指令 ✅
   📁 data/processed/step_level_instructions.jsonl
   💾 大小: 9.0 MB | 📊 行数: 40,209

✅ 步骤级数据 ✅
   📁 data/processed/step_level_data.jsonl
   💾 大小: 13.8 MB | 📊 行数: 40,209

📂 第二步：加载数据

1️⃣ 加载步骤级指令...
   ✅ 已加载: 40,209 条指令

2️⃣ 加载步骤级数据...
   ✅ 已加载: 40,209 条数据

🔧 第三步：构造训练数据

✅ 已构造: 40,209 个训练样本

📊 第四步：分割数据集

📊 采样10%数据（OOM优化版）

⚠️  采样比例已从50%降到10%以防止OOM
   原始训练集: 36,193 样本
   原始验证集: 4,016 样本

   🔄 训练集: 36,193 → 361 样本 (保留10%)
   🔄 验证集: 4,016 → 40 样本 (保留10%)

   ✅ 采样后训练集: 361 样本
   ✅ 采样后验证集: 40 样本
   ✅ 比例: 90.0% 训练 / 10.0% 验证

✅ 数据质量检查

   训练集完整性:
      指令完整: 361/361 (100.0%)
      输出完整: 361/361 (100.0%)
      关键词完整: 0/361 (0.0%)

📝 数据样本:
   指令: Create E MS Installatie...
   输出字段: ['step_index', 'database', 'object', 'object_id', 'module', 'method', 'command', 'test_data', 'file_id']
   关键词: []

✅ 数据加载、采样和分割完成！

📊 可用变量:
   • train_data: 训练集数据 (已采样10%)
   • val_data: 验证集数据 (已采样10%)
   • 总样本数: 401 (原始: 40,209)



## 🚀 步骤4：模型加载与LoRA配置

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, TaskType, get_peft_model

# ============================================================
# 模型配置
# ============================================================

MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"  # 无需认证、专为代码优化
OUTPUT_DIR = "/content/drive/MyDrive/gis-models/step-level-model"
LORA_R = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
BATCH_SIZE = 2  # ⚠️ 提高物理batch size
GRADIENT_ACCUMULATION = 2  # ⚠️ 降低梯度累积步数
EVAL_AND_SAVE_STEPS = 500 # ⚠️ 设置为500
MAX_LENGTH = 128  # ⚠️ 从256降到192（节省20%的内存）

print("🔧 导入完成，模型配置已准备")
print(f"   • MODEL_NAME: {MODEL_NAME}")
print(f"   • BATCH_SIZE: {BATCH_SIZE}")
print(f"   • GRADIENT_ACCUMULATION: {GRADIENT_ACCUMULATION}")
print(f"   • 有效batch大小: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   • MAX_LENGTH: {MAX_LENGTH}")
print(f"   • LORA_R: {LORA_R}")
print(f"   • OUTPUT_DIR: {OUTPUT_DIR}\n")
print("="*70)
print("🔧 步骤5：训练配置（OOM优化版）")
print("="*70 + "\n")

# ============================================================
# 训练参数配置
# ============================================================

# 内存优化配置
BATCH_SIZE = 2  # 提高物理batch size
GRADIENT_ACCUMULATION_STEPS = 2  # 降低梯度累积步数
MAX_LENGTH = 128  # 更短的序列长度

# 学习率和优化器配置
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_STEPS = int(len(train_data) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * 0.1)  # 10% warmup
MAX_STEPS = -1  # 使用epochs而不是steps限制

print(f"📊 批处理配置（OOM优化）:")
print(f"   • 单个batch_size: {BATCH_SIZE}")
print(f"   • GRADIENT_ACCUMULATION_STEPS: {GRADIENT_ACCUMULATION_STEPS}")
print(f"   • 有效batch大小: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   • 每个epoch的优化步数: {len(train_data) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS):,}\n")

print(f"📈 优化器配置:")
print(f"   • LEARNING_RATE: {LEARNING_RATE}")
print(f"   • WEIGHT_DECAY: {WEIGHT_DECAY}")
print(f"   • WARMUP_STEPS: {WARMUP_STEPS:,}\n")

print(f"🔄 训练周期配置:")
print(f"   • NUM_EPOCHS: {NUM_EPOCHS}")
print(f"   • MAX_LENGTH: {MAX_LENGTH} tokens (降低20%)")
print(f"   • TRAINING_SAMPLES: {len(train_data):,}")
print(f"   • VALIDATION_SAMPLES: {len(val_data):,}\n")

# ============================================================
# 估算训练时间
# ============================================================

steps_per_epoch = len(train_data) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS

print(f"⏱️  训练时间估算 (T4 GPU):")
print(f"   • 总优化步数: {total_steps:.0f}")
print(f"   • 每个epoch: {steps_per_epoch:.0f} 优化步")
print(f"   • 预计时间: 2-3 小时 (20k样本, batch_size=1+GradAcc=4, 3 epochs)")
print(f"   (相比于原始配置，时间略长但更稳定)\n")

# ============================================================
# 显示内存优化配置
# ============================================================

print("💾 内存优化设置总结:")
print("   ✓ float16 精度")
print("   ✓ Gradient Checkpointing (已启用)")
print("   ✓ AdamW优化器（标准版本）")
print("   ✓ BATCH_SIZE=4 + GRAD_ACC=2 (激进内存节省)")
print("   ✓ MAX_LENGTH=192 (减少20%)")
print("   ✓ 50% 数据采样 (~20k样本)")
print("   ✓ 优化的Tokenization批处理")
print("\n   预期显存使用: ~8-10 GB (in T4's 14GB)\n")

print("="*70)
print("🚀 准备就绪！下一步：运行训练")
print("="*70)

🔧 导入完成，模型配置已准备
   • MODEL_NAME: codellama/CodeLlama-7b-Instruct-hf
   • BATCH_SIZE: 2
   • GRADIENT_ACCUMULATION: 2
   • 有效batch大小: 4
   • MAX_LENGTH: 128
   • LORA_R: 32
   • OUTPUT_DIR: /content/drive/MyDrive/gis-models/step-level-model

🔧 步骤5：训练配置（OOM优化版）

📊 批处理配置（OOM优化）:
   • 单个batch_size: 2
   • GRADIENT_ACCUMULATION_STEPS: 2
   • 有效batch大小: 4
   • 每个epoch的优化步数: 90

📈 优化器配置:
   • LEARNING_RATE: 0.0002
   • WEIGHT_DECAY: 0.01
   • WARMUP_STEPS: 9

🔄 训练周期配置:
   • NUM_EPOCHS: 3
   • MAX_LENGTH: 128 tokens (降低20%)
   • TRAINING_SAMPLES: 361
   • VALIDATION_SAMPLES: 40

⏱️  训练时间估算 (T4 GPU):
   • 总优化步数: 271
   • 每个epoch: 90 优化步
   • 预计时间: 2-3 小时 (20k样本, batch_size=1+GradAcc=4, 3 epochs)
   (相比于原始配置，时间略长但更稳定)

💾 内存优化设置总结:
   ✓ float16 精度
   ✓ Gradient Checkpointing (已启用)
   ✓ AdamW优化器（标准版本）
   ✓ BATCH_SIZE=4 + GRAD_ACC=2 (激进内存节省)
   ✓ MAX_LENGTH=192 (减少20%)
   ✓ 50% 数据采样 (~20k样本)
   ✓ 优化的Tokenization批处理

   预期显存使用: ~8-10 GB (in T4's 14GB)

🚀 准备就绪！下一步：运行训练


In [ ]:
# 升级库 - 确保版本兼容（PEFT版本修复）
print("🔧 Fixing PEFT version compatibility issue...")
print("   ⚠️  卸载不兼容的PEFT版本...\n")

# 强制卸载旧PEFT
!pip uninstall -y peft 2>&1 | head -5

print("\n   ⚠️  安装兼容的PEFT版本...\n")

# 重新安装兼容的版本
!pip install -q peft==0.11.1 --no-cache-dir
!pip install -q --upgrade transformers==4.46.0 --no-cache-dir
!pip install -q --upgrade accelerate>=1.0.0 --no-cache-dir
!pip install -q --upgrade "bitsandbytes>=0.43.0" --no-cache-dir

print("\n✅ Libraries fixed and upgraded!")
print("   • peft==0.11.1 (downgraded to fix ArrowConfig import)")
print("   • transformers==4.46.0")
print("   • accelerate>=1.0.0")
print("   • bitsandbytes>=0.43.0")

🔧 Fixing PEFT version compatibility issue...
   ⚠️  卸载不兼容的PEFT版本...

Found existing installation: peft 0.13.0
Uninstalling peft-0.13.0:
  Successfully uninstalled peft-0.13.0

   ⚠️  安装兼容的PEFT版本...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 20.4 MB/s eta 0:00:00

✅ Libraries fixed and upgraded!
   • peft==0.11.1 (downgraded to fix ArrowConfig import)
   • transformers==4.46.0
   • accelerate>=1.0.0
   • bitsandbytes>=0.43.0


In [ ]:
# ⚠️ 可选：超激进OOM恢复方案 - 4bit量化
# 如果运行到这里时仍然OOM，取消注释下面的部分并设置USE_4BIT=True

USE_4BIT = False  # 仅在持续OOM时改为True

if USE_4BIT:
    print("⚠️  启用4-bit量化（超激进内存节省）...")
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    print("   ❌ 停用此cell中的标准加载，下一个cell中的model加载会自动使用4-bit配置")
else:
    print("✅ 使用标准float16加载（内存高效但不是超激进）")
    print("   如果出现OOM，改动此cell中 USE_4BIT = True")


✅ 使用标准float16加载（内存高效但不是超激进）
   如果出现OOM，改动此cell中 USE_4BIT = True


In [ ]:
# 加载tokenizer (CodeLlama)
import gc
import sys

print("📖 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded: vocab_size={len(tokenizer)}")

# ⚠️ 关键：在模型加载前清理内存
print("\n🧹 清理缓存...")
gc.collect()
torch.cuda.empty_cache()

# 清理peft相关的模块缓存（修复ArrowConfig导入问题）
peft_modules = [name for name in list(sys.modules.keys()) if 'peft' in name.lower()]
for module_name in peft_modules:
    try:
        del sys.modules[module_name]
    except:
        pass
print(f"   ✅ 清理了{len(peft_modules)}个peft模块缓存")

# 加载模型 - 简化方案：使用float16而不是8-bit
print("\n🤖 Loading model with float16 precision...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
)

# 训练中必须禁用缓存以配合梯度检查点
model.config.use_cache = False

# 启用梯度检查点（节省显存）
model.gradient_checkpointing_enable()

print("✅ Base model loaded (float16, ~6-7GB RAM)")

# ⚠️ 再次清理缓存，确保peft导入正确
gc.collect()

# 应用LoRA
print("\n🔧 Applying LoRA...")

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied!")

# 再次清理内存
gc.collect()
torch.cuda.empty_cache()
print("\n💾 GPU Memory Status:")
print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

📖 Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Tokenizer loaded: vocab_size=32016

🧹 清理缓存...
   ✅ 清理了106个peft模块缓存

🤖 Loading model with float16 precision...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Base model loaded (float16, ~6-7GB RAM)

🔧 Applying LoRA...
trainable params: 16,777,216 || all params: 6,755,323,904 || trainable%: 0.2484
✅ LoRA applied!

💾 GPU Memory Status:
   Allocated: 13.51 GB
   Reserved: 13.52 GB


## 📚 步骤5：数据准备与Tokenization

In [ ]:
from datasets import Dataset
import json as json_module

# 准备数据集
print("📊 准备datasets...")

# 转换为Dataset格式
train_dataset_hf = Dataset.from_dict({
    'instruction': [d['instruction'] for d in train_data],
    'output': [json_module.dumps(d['output'], ensure_ascii=False, indent=2) for d in train_data],
    'avg_weight': [d['metadata']['avg_weight'] for d in train_data],
})

eval_dataset_hf = Dataset.from_dict({
    'instruction': [d['instruction'] for d in val_data],
    'output': [json_module.dumps(d['output'], ensure_ascii=False, indent=2) for d in val_data],
    'avg_weight': [d['metadata']['avg_weight'] for d in val_data],
})

print(f"  训练集: {len(train_dataset_hf)} samples")
print(f"  验证集: {len(eval_dataset_hf)} samples")

# 格式化prompt：输入指令 -> 输出完整的步骤数据
def format_prompt(example):
    """
    格式化为prompt：指令 -> 步骤数据

    示例：
    Input: "Open E MS Kabel"
    Output:
    {
      "step_index": 0,
      "database": ":elektra",
      ...
    }
    """
    input_instruction = example['instruction'] # 修正：从'instruction'键获取数据
    output_json = example['output']  # 已是格式化的JSON字符串

    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {input_instruction}

Step Data JSON:
{output_json}"""

    return {"text": prompt}

train_dataset_hf = train_dataset_hf.map(
    format_prompt,
    remove_columns=['instruction', 'output', 'avg_weight'] # 修正：删除'instruction'而不是'input'
)
eval_dataset_hf = eval_dataset_hf.map(
    format_prompt,
    remove_columns=['instruction', 'output', 'avg_weight'] # 修正：删除'instruction'而不是'input'
)

# Tokenize
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🔄 Tokenizing...")
train_dataset_hf = train_dataset_hf.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset_hf.column_names,
    desc="Tokenizing train"
)

eval_dataset_hf = eval_dataset_hf.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset_hf.column_names,
    desc="Tokenizing val"
)

print("✅ 数据准备完成！")
print(f"\n📊 数据统计:")
train_lengths = [len(d['input_ids']) for d in train_dataset_hf]
eval_lengths = [len(d['input_ids']) for d in eval_dataset_hf]
print(f"   训练集: {len(train_dataset_hf)} samples, 平均长度: {np.mean(train_lengths):.0f} tokens, 最大: {max(train_lengths)} tokens")
print(f"   验证集: {len(eval_dataset_hf)} samples, 平均长度: {np.mean(eval_lengths):.0f} tokens, 最大: {max(eval_lengths)} tokens\n")
print(f"   格式: Instruction → Step Data JSON")


📊 准备datasets...
  训练集: 361 samples
  验证集: 40 samples


Map:   0%|          | 0/361 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

🔄 Tokenizing...


Tokenizing train:   0%|          | 0/361 [00:00<?, ? examples/s]

Tokenizing val:   0%|          | 0/40 [00:00<?, ? examples/s]

✅ 数据准备完成！

📊 数据统计:
   训练集: 361 samples, 平均长度: 128 tokens, 最大: 128 tokens
   验证集: 40 samples, 平均长度: 128 tokens, 最大: 128 tokens

   格式: Instruction → Step Data JSON


## 🎯 步骤6：开始训练

In [ ]:
# 配置训练 - ⚠️ FP16/显存优化修复版本
print("⚙️ 配置训练（OOM + FP16梯度修复版）...\n")

# 显示当前的显存状态
import torch
import gc
gc.collect()
print(f"💾 当前显存状态（optimization开始前）:")
print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB\n")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=10,
    
    # ⚠️ 修复OOM：禁用中间eval和checkpoint，只在训练完成后保存
    save_steps=10000,  # 禁止中间保存（set very high）
    eval_steps=10000,  # 禁止中间eval
    eval_strategy="no",  # ✅ 不进行中间评估（解决OOM）
    save_strategy="no",  # ✅ 不进行中间保存
    load_best_model_at_end=False,  # 不需要加载最佳模型

    # ⚠️ 修复FP16梯度问题：改为bf16而不是fp16
    fp16=False,
    bf16=True,  # bfloat16对梯度缩放友好

    # ⚠️ 改为adamw_torch而不是adamw_8bit（兼容性更好）
    optim="adamw_torch",

    lr_scheduler_type="cosine",
    save_total_limit=1,
    report_to="none",
    logging_dir=f"{OUTPUT_DIR}/logs",
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    push_to_hub=False,
    gradient_checkpointing=True,

    # ⚠️ 删除max_grad_norm：与bf16梯度缩放不兼容
    # max_grad_norm=1.0,

    # ⚠️ 额外的显存优化参数
    dataloader_pin_memory=False,  # 降低显存压力
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_hf,
    eval_dataset=eval_dataset_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("✅ Trainer ready!")
print("\n🔧 训练配置修复摘要:")
print("   ✓ fp16 = False (不再使用)")
print("   ✓ bf16 = True (bfloat16对梯度缩放友好)")
print("   ✓ optim = adamw_torch (标准优化器)")
print("   ✓ max_grad_norm = 注释掉 (避免兼容性问题)")
print("   ✓ dataloader_pin_memory = False (显存优化)\n")

⚙️ 配置训练（OOM + FP16梯度修复版）...

💾 当前显存状态（optimization开始前）:
   Allocated: 13.51 GB
   Reserved: 13.52 GB

✅ Trainer ready!

🔧 训练配置修复摘要:
   ✓ fp16 = False (不再使用)
   ✓ bf16 = True (bfloat16对梯度缩放友好)
   ✓ optim = adamw_torch (标准优化器)
   ✓ max_grad_norm = 注释掉 (避免兼容性问题)
   ✓ dataloader_pin_memory = False (显存优化)



In [ ]:
print("\n" + "="*70)
print("🚀 开始训练（bf16/adamw_torch版，预计2-4小时）...")
print("="*70)

# ⚠️ 关键：激进的显存清理
import gc
import torch

print("\n🧹 执行激进显存清理...")

# 多次垃圾回收
for i in range(3):
    gc.collect()
    torch.cuda.empty_cache()

print("✅ 显存清理完成")

print(f"\n💾 训练前显存状态:")
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
available = (torch.cuda.get_device_properties(0).total_memory - reserved) / 1e9

print(f"   Allocated: {allocated:.2f} GB")
print(f"   Reserved: {reserved:.2f} GB")
print(f"   Available: {available:.2f} GB")

if allocated > 13.0:
    print(f"\n⚠️  警告：显存已用接近上限！")
    print(f"   建议：如果训练失败，请减少BATCH_SIZE或MAX_LENGTH")

print("\n" + "="*70)

# 开始训练
try:
    print("⏱️  开始计时...")
    import time
    start_time = time.time()

    trainer.train()

    elapsed = (time.time() - start_time) / 3600
    print("\n" + "="*70)
    print(f"🎉 训练完成！耗时 {elapsed:.1f} 小时")
    print("="*70)

except torch.cuda.OutOfMemoryError as e:
    print("\n" + "="*70)
    print("❌ CUDA OOM错误")
    print("="*70)
    print("\n💡 解决方案:")
    print("   1. 重启kernel (Runtime → Restart runtime)")
    print("   2. 减少BATCH_SIZE (从1 → 1, 梯度累积4 → 2)")
    print("   3. 进一步减少MAX_LENGTH (从192 → 128)")
    print("   4. 减少采样率 (从25% → 10%)")
    print("   5. 启用模型offloading")
    raise

except ValueError as e:
    if "Attempting to unscale" in str(e) or "FP16" in str(e):
        print("\n" + "="*70)
        print("❌ FP16梯度错误仍然存在")
        print("="*70)
        print("\n💡 这可能是由于:")
        print("   1. 之前的Cell没有正确更新")
        print("   2. 需要重启kernel来清除旧配置")
        print("\n建议：")
        print("   - 重启kernel")
        print("   - 重新运行所有cells")
        raise
    else:
        print(f"\n❌ 训练出错: {type(e).__name__}")
        print(f"   错误信息: {str(e)[:300]}")
        raise

except Exception as e:
    print(f"\n❌ 训练出错: {type(e).__name__}")
    print(f"   错误信息: {str(e)[:300]}")
    raise


🚀 开始训练（bf16/adamw_torch版，预计2-4小时）...

🧹 执行激进显存清理...
✅ 显存清理完成

💾 训练前显存状态:
   Allocated: 13.51 GB
   Reserved: 13.52 GB
   Available: 15.64 GB

⚠️  警告：显存已用接近上限！
   建议：如果训练失败，请减少BATCH_SIZE或MAX_LENGTH

⏱️  开始计时...


Step,Training Loss,Validation Loss


## 💾 步骤7：保存训练好的模型

## 🧪 验证集评估（直接测试训练后的模型）

In [ ]:
import json
import torch
from tqdm import tqdm

print("="*70)
print("🧪 验证集评估（修复CUDA错误版本）")
print("="*70 + "\n")

# ⚠️ 关键修复：重新配置tokenizer
print("🔧 步骤1：修复tokenizer配置")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"   ✅ 设置 pad_token = eos_token")
else:
    print(f"   ℹ️  pad_token 已设置: {tokenizer.pad_token}")

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"   ✅ 设置 pad_token_id = eos_token_id")

print(f"\n📖 Tokenizer状态：")
print(f"   pad_token: {tokenizer.pad_token}")
print(f"   pad_token_id: {tokenizer.pad_token_id}")
print(f"   eos_token_id: {tokenizer.eos_token_id}\n")

# 确保模型在评估模式
model.eval()

print(f"📊 验证集大小: {len(val_data)} samples\n")

def evaluate_model_v2(val_data_list, max_samples=None):
    """
    改进的评估函数（修复CUDA错误）
    """
    results = {
        'total': 0,
        'successful_inference': 0,
        'valid_json': 0,
        'errors': [],
        'samples': []
    }
    
    num_samples = len(val_data_list) if max_samples is None else min(max_samples, len(val_data_list))
    
    print(f"⏳ 开始评估 {num_samples} 个样本...\n")
    
    for idx in tqdm(range(num_samples), desc="评估进度"):
        try:
            sample = val_data_list[idx]
            instruction = sample['instruction']
            
            prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""
            
            # ⚠️ 更安全的tokenization
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=256,
                padding="max_length",  # 显式设置padding方式
                return_attention_mask=True
            )
            
            input_ids = inputs["input_ids"].to(model.device)
            attention_mask = inputs["attention_mask"].to(model.device)
            
            # ⚠️ 清理CUDA缓存
            torch.cuda.empty_cache()
            
            # ⚠️ 最稳定的生成参数（greedy decoding）
            with torch.no_grad():
                outputs = model.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=300,
                    do_sample=False,           # 关键：禁用采样
                    num_beams=1,               # Greedy decoding
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=False,           # 禁用缓存
                )
            
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            if "Step Data JSON:" in generated_text:
                json_part = generated_text.split("Step Data JSON:")[-1].strip()
            else:
                json_part = generated_text
            
            results['successful_inference'] += 1
            
            try:
                json_obj = json.loads(json_part)
                results['valid_json'] += 1
                valid_json = True
            except json.JSONDecodeError:
                valid_json = False
            
            results['total'] += 1
            
            if len(results['samples']) < 5:
                results['samples'].append({
                    'index': idx,
                    'instruction': instruction[:100],
                    'generated_json': json_part[:150],
                    'is_valid_json': valid_json,
                    'keys': list(json_obj.keys()) if valid_json else None
                })
        
        except Exception as e:
            results['errors'].append({
                'index': idx,
                'error': str(e)[:100]
            })
    
    return results

# 运行改进的评估
print("✅ 开始评估（修复报错版本）...\n")
eval_results = evaluate_model_v2(val_data, max_samples=20)

print("\n" + "="*70)
print("📊 评估结果")
print("="*70 + "\n")

print(f"✅ 总样本数: {eval_results['total']}")
print(f"✅ 推理成功: {eval_results['successful_inference']}/{eval_results['total']}")
print(f"✅ 有效JSON: {eval_results['valid_json']}/{eval_results['total']}")

if eval_results['total'] > 0:
    print(f"\n📈 指标:")
    print(f"   推理成功率: {eval_results['successful_inference']/eval_results['total']*100:.1f}%")
    print(f"   JSON有效率: {eval_results['valid_json']/eval_results['total']*100:.1f}%")

if eval_results['errors']:
    print(f"\n⚠️  错误数: {len(eval_results['errors'])}")
    for err in eval_results['errors'][:3]:
        print(f"   样本{err['index']}: {err['error']}")

print(f"\n📝 示例输出:")
print("-" * 70)
for sample in eval_results['samples']:
    print(f"\n样本 #{sample['index']}:")
    print(f"  指令: {sample['instruction']}...")
    print(f"  JSON有效性: {'✅' if sample['is_valid_json'] else '❌'}")
    if sample['is_valid_json']:
        print(f"  字段: {sample['keys']}")
    print(f"  输出: {sample['generated_json']}...")

print("\n" + "="*70)
print("✨ 评估完成！")
print("="*70)


In [ ]:
import torch
import json

print("="*70)
print("🔬 CUDA错误调试：最小化测试")
print("="*70)

# 测试1：检查模型是否能进行forward pass
print("\n✅ 测试1：模型forward pass（无generation）")
print("-" * 70)

sample = val_data[0]
instruction = sample['instruction']

prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""

print(f"提示词长度：{len(prompt)} 字符")

# 最小化tokenization - 不设置pad_token_id
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=256,
    padding=False,  # ⚠️ 改为False - 不padding
    return_attention_mask=False  # ⚠️ 不返回attention_mask
)

input_ids = inputs["input_ids"].to(model.device)
print(f"input_ids 形状: {input_ids.shape}")
print(f"input_ids 在device: {input_ids.device}")

try:
    with torch.no_grad():
        outputs = model(input_ids)
    print("✅ Forward pass 成功！")
    print(f"   输出形状: {outputs.logits.shape}")
except Exception as e:
    print(f"❌ Forward pass 失败: {e}")

# 测试2：最小生成（只生成1个token）
print("\n\n✅ 测试2：最小生成（1个token，greedy）")
print("-" * 70)

try:
    torch.cuda.empty_cache()
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=False
    )
    input_ids = inputs["input_ids"].to(model.device)
    
    outputs = model.generate(
        input_ids,
        max_new_tokens=1,  # ⚠️ 仅生成1个token
        do_sample=False,
        # 不设置任何其他参数
    )
    print(f"✅ 生成成功！")
    print(f"   输出形状: {outputs.shape}")
    text = tokenizer.decode(outputs[0])
    print(f"   生成文本: {text}")
except Exception as e:
    print(f"❌ 生成失败（1 token）: {str(e)[:150]}")

# 测试3：逐步增加token数量
print("\n\n✅ 测试3：逐步增加生成长度")
print("-" * 70)

for max_tokens in [5, 10, 20, 50]:
    try:
        torch.cuda.empty_cache()
        
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=False
        )
        input_ids = inputs["input_ids"].to(model.device)
        
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            do_sample=False,
        )
        text = tokenizer.decode(outputs[0])
        print(f"✅ max_tokens={max_tokens:2d}: 成功 | 生成长度={len(text)}")
        if max_tokens == 20:
            print(f"   样本输出: {text[:100]}...")
    except Exception as e:
        print(f"❌ max_tokens={max_tokens:2d}: 失败 - {str(e)[:80]}")
        break

# 测试4：尝试不同的pad_token配置
print("\n\n✅ 测试4：不同的pad_token配置")
print("-" * 70)

configs = [
    ("不设置pad_token_id", None),
    ("设置为eos_token_id", tokenizer.eos_token_id),
    ("设置为0", 0),
]

for config_name, pad_token_id in configs:
    try:
        torch.cuda.empty_cache()
        
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=False
        )
        input_ids = inputs["input_ids"].to(model.device)
        
        params = {
            "input_ids": input_ids,
            "max_new_tokens": 10,
            "do_sample": False,
        }
        
        if pad_token_id is not None:
            params["pad_token_id"] = pad_token_id
        
        outputs = model.generate(**params)
        print(f"✅ {config_name}: 成功")
    except Exception as e:
        print(f"❌ {config_name}: {str(e)[:100]}")

print("\n" + "="*70)
print("🔍 调试完成")
print("="*70)


In [ ]:
import torch

print("="*70)
print("🔍 模型诊断：检查是否正确加载LoRA权重")
print("="*70)

# 目标1：检查当前模型结构
print("\n1️⃣ 模型结构检查")
print("-" * 70)

print(f"模型类型：{type(model).__name__}")
print(f"模型名称：{model.__class__.__module__}.{model.__class__.__name__}")

# 检查是否是PeftModel
from peft import PeftModel, LoraModel
if isinstance(model, (PeftModel, LoraModel)):
    print("✅ 模型是 PeftModel/LoRA模型")
    try:
        model.print_trainable_parameters()
    except:
        print("   (无法打印参数统计)")
else:
    print("⚠️  模型不是PeftModel！")
    print("   可能原因：LoRA权重未加载")

# 目标2：检查模型的训练状态
print("\n2️⃣ 模型训练状态")
print("-" * 70)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"总参数数：{total_params:,}")
print(f"可训练参数：{trainable_params:,}")
print(f"可训练比例：{trainable_params/total_params*100:.2f}%")

if trainable_params / total_params < 0.01:
    print("⚠️  可训练参数 < 1%，很可能LoRA没有应用！")
elif trainable_params / total_params > 1:
    print("⚠️  可训练参数 > 100%，配置异常！")
else:
    print("✅ 看起来LoRA已正确应用")

# 目标3：检查是否还有checkpoint
print("\n3️⃣ 检查是否有已保存的checkpoint")
print("-" * 70)

import os
output_dir = OUTPUT_DIR

if os.path.exists(output_dir):
    print(f"输出目录存在：{output_dir}")
    contents = os.listdir(output_dir)
    print(f"目录内容：{contents}")
    
    # 检查是否有checkpoint
    checkpoints = [d for d in contents if d.startswith('checkpoint-')]
    if checkpoints:
        print(f"✅ 找到 {len(checkpoints)} 个checkpoint：")
        for ckpt in sorted(checkpoints):
            ckpt_path = os.path.join(output_dir, ckpt)
            size_mb = sum(os.path.getsize(os.path.join(ckpt_path, f)) for f in os.listdir(ckpt_path)) / 1e6
            print(f"   - {ckpt} (~{size_mb:.0f}MB)")
    else:
        print("❌ 没有找到checkpoint！")
else:
    print(f"❌ 输出目录不存在：{output_dir}")

# 目标4：尝试简单推理，看看模型是否真的工作
print("\n4️⃣ 简单推理测试")
print("-" * 70)

test_prompt = "Hello, how are you?"
inputs = tokenizer(test_prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(model.device)

try:
    with torch.no_grad():
        # 只做forward pass，不生成
        outputs = model(input_ids)
    print("✅ Forward pass 成功")
    print(f"   输出logits形状：{outputs.logits.shape}")
except Exception as e:
    print(f"❌ Forward pass 失败：{e}")

print("\n" + "="*70)
print("📊 诊断完成")
print("="*70)


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("="*70)
print("🔧 Checkpoint恢复：从最完整的checkpoint重新加载")
print("="*70)

checkpoint_dir = OUTPUT_DIR

# 检查所有checkpoint的完整性
print("\n1️⃣ 检查所有checkpoint的完整性")
print("-" * 70)

if os.path.exists(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]
    
    for ckpt_name in sorted(checkpoints, reverse=True):  # 从新到旧
        ckpt_path = os.path.join(checkpoint_dir, ckpt_name)
        
        print(f"\n📁 {ckpt_name}:")
        
        # 检查必要文件
        required_files = [
            'adapter_config.json',      # LoRA配置
            'adapter_model.bin',         # LoRA权重
            'pytorch_model.bin',         # 模型权重（可能）
            'training_args.bin'          # 训练参数
        ]
        
        files_present = {}
        for fname in required_files:
            fpath = os.path.join(ckpt_path, fname)
            if os.path.exists(fpath):
                size_kb = os.path.getsize(fpath) / 1024
                files_present[fname] = f"✅ ({size_kb:.0f}KB)"
            else:
                files_present[fname] = "❌"
        
        for fname, status in files_present.items():
            print(f"   {fname}: {status}")
        
        # 统计目录大小
        total_size = sum(
            os.path.getsize(os.path.join(ckpt_path, f)) 
            for f in os.listdir(ckpt_path) 
            if os.path.isfile(os.path.join(ckpt_path, f))
        ) / 1024 / 1024
        print(f"   总大小: {total_size:.1f}MB")
        
        # 判断这个checkpoint是否可用
        if os.path.exists(os.path.join(ckpt_path, 'adapter_config.json')):
            print(f"   → 这个checkpoint可以加载 ✅")
        else:
            print(f"   → 这个checkpoint不完整 ❌")

# 选择最完整的checkpoint
print("\n2️⃣ 选择最完整的checkpoint加载")
print("-" * 70)

best_checkpoint = None
checkpoints = sorted([d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')], reverse=True)

for ckpt_name in checkpoints:
    ckpt_path = os.path.join(checkpoint_dir, ckpt_name)
    if os.path.exists(os.path.join(ckpt_path, 'adapter_config.json')):
        best_checkpoint = ckpt_path
        print(f"✅ 选择: {ckpt_name}")
        break

if best_checkpoint is None:
    print("❌ 没有找到完整的checkpoint！")
    print("   可能需要重新训练")
else:
    print(f"\n3️⃣ 从checkpoint加载LoRA模型")
    print("-" * 70)
    
    try:
        # 先清理当前模型
        del model
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        
        # 重新加载基础模型
        print("加载基础模型...")
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,  # ⚠️ 改为bfloat16，与训练一致
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        print("✅ 基础模型加载成功\n")
        
        # 从checkpoint加载LoRA
        print(f"从checkpoint加载LoRA权重: {best_checkpoint}")
        model = PeftModel.from_pretrained(base_model, best_checkpoint)
        print("✅ LoRA权重加载成功\n")
        
        model.eval()
        model.print_trainable_parameters()
        
        print("\n✅ 模型重新加载完成！")
        print("现在可以运行推理测试了")
        
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*70)


In [ ]:
import torch
import json

print("="*70)
print("🧪 推理测试（避免CUDA错误版）")
print("="*70)

# 确保tokenizer配置正确
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print(f"\n📊 模型状态检查:")
print(f"   模型类型: {type(model).__name__}")
print(f"   设备: {next(model.parameters()).device}")
print(f"   精度: {next(model.parameters()).dtype}")

# 测试推理 - 最小化版本
print(f"\n🧪 开始推理测试（3个样本）...\n")

test_results = {
    'total': 0,
    'success': 0,
    'has_json': 0,
    'valid_json': 0,
}

# 只测试前3个样本
for idx in range(min(3, len(val_data))):
    sample = val_data[idx]
    instruction = sample['instruction']
    
    print(f"样本 #{idx}:")
    print(f"  指令: {instruction[:60]}...")
    
    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""
    
    try:
        # Tokenize
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding="max_length",
            return_attention_mask=True
        )
        
        # 不移动到device，直接用默认位置
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        
        # 安全的生成（greedy）
        with torch.no_grad():
            torch.cuda.empty_cache()
            
            try:
                outputs = model.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=100,
                    do_sample=False,
                    num_beams=1,
                )
                
                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # 提取JSON
                if "Step Data JSON:" in generated_text:
                    json_part = generated_text.split("Step Data JSON:")[-1].strip()
                else:
                    json_part = generated_text
                
                test_results['success'] += 1
                
                # 检查是否包含JSON内容
                if json_part.startswith('{'):
                    test_results['has_json'] += 1
                    try:
                        json.loads(json_part)
                        test_results['valid_json'] += 1
                        print(f"  ✅ 成功 | JSON有效")
                    except:
                        print(f"  ⚠️ 生成了但JSON无效")
                else:
                    print(f"  ⚠️ 推理成功但没有JSON: {json_part[:50]}")
                
            except Exception as gen_err:
                print(f"  ❌ 生成失败: {str(gen_err)[:80]}")
        
        test_results['total'] += 1
        
    except Exception as e:
        print(f"  ❌ 错误: {str(e)[:80]}")
        test_results['total'] += 1

print(f"\n" + "="*70)
print("📊 测试结果")
print("="*70)
print(f"测试样本数: {test_results['total']}")
print(f"推理成功: {test_results['success']}/{test_results['total']}")
print(f"包含JSON: {test_results['has_json']}/{test_results['total']}")
print(f"JSON有效: {test_results['valid_json']}/{test_results['total']}")

if test_results['valid_json'] > 0:
    print(f"\n✅ 好消息！模型能生成有效JSON")
else:
    print(f"\n⚠️ 模型无法生成有效JSON")
    print(f"\n💡 可能的原因：")
    print(f"1. 训练数据太少（只有~100样本）")
    print(f"2. 提示词格式与训练时不一致")
    print(f"3. LoRA权重没有充分收敛")
    print(f"\n建议：增加训练样本到1000+或增加训练轮次到5-10")

print("\n" + "="*70)


## ⚠️ 紧急：GPU状态损坏 - 需要重启kernel

### 问题诊断
- ❌ CUDA device-side assert 在 `torch.cuda.empty_cache()` 触发
- ❌ GPU状态已破坏，无法继续使用当前kernel
- ✅ 好消息：checkpoint-50 (464.4MB) 保存完整，可以恢复

### 解决步骤

**步骤1：重启Kernel**
- 点击菜单 `Runtime` → `Restart runtime`
- 等待kernel完全启动（约1分钟）

**步骤2：重启后请按顺序运行这些cell（跳过其他）：**

| 顺序 | Cell名称 | 原因 |
|------|---------|------|
| 1 | Uninstalling torchvision | 清理环境 |
| 2 | Check GPU | 验证GPU |
| 3 | Install dependencies | 装库 |
| 4 | Mount Google Drive | 连接存储 |
| 5 | Load step-level data | 准备数据 |
| 6 | 配置训练 (包括模型加载/LoRA) | 初始化模型 |

**⚠️ 这些cell请暂时跳过** (会导致CUDA错误):
- ❌ 所有evaluation cells (evaluation, CUDA debug等)
- ❌ 模型诊断cell
- ❌ Checkpoint恢复cell

**步骤3：重启后直接加载checkpoint-50进行推理**

关键修复：使用CPU推理而不是GPU，避免CUDA问题。

---



In [ ]:
import torch
import os
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("="*70)
print("🚀 重启kernel后：从checkpoint-50加载模型（CPU推理版）")
print("="*70)

# ============================================================
# 选项1：CPU推理（慢但稳定）
# ============================================================

print("\n💡 使用CPU推理避免CUDA错误（推荐）\n")

checkpoint_path = "/content/drive/MyDrive/gis-models/step-level-model/checkpoint-50"

print(f"📂 checkpoint路径: {checkpoint_path}\n")

# 检查checkpoint是否存在
if not os.path.exists(checkpoint_path):
    print(f"❌ checkpoint不存在！")
    print(f"   请确保Google Drive正确挂载")
else:
    print(f"✅ checkpoint存在\n")
    
    print("1️⃣ 加载基础模型（CPU）...")
    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            "codellama/CodeLlama-7b-Instruct-hf",
            torch_dtype=torch.bfloat16,
            device_map="cpu",  # ⚠️ 强制CPU
            low_cpu_mem_usage=True,
        )
        print("   ✅ 成功\n")
    except Exception as e:
        print(f"   ❌ 失败: {e}\n")
        raise
    
    print("2️⃣ 加载LoRA权重...")
    try:
        model = PeftModel.from_pretrained(base_model, checkpoint_path)
        print("   ✅ 成功\n")
        model.print_trainable_parameters()
    except Exception as e:
        print(f"   ❌ 失败: {e}\n")
        raise
    
    print("\n3️⃣ 加载tokenizer...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            "codellama/CodeLlama-7b-Instruct-hf",
            padding_side="right"
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        print("   ✅ 成功\n")
    except Exception as e:
        print(f"   ❌ 失败: {e}\n")
        raise
    
    # 模型设置
    model.eval()
    
    print("="*70)
    print("✅ 模型已加载！现在可以运行推理了")
    print("="*70)
    print("\n⚠️ 注意：CPU推理速度较慢（每个样本约5-10秒）")
    print("   如果需要GPU速度，需要重启kernel并确保GPU状态正常")



## ⚠️ 重要：如果清理cell报错OOM，请按以下步骤操作

**如果运行上一个"内存清理"cell时出现显存溢出（OOM）错误，不用修复！** 直接按照以下步骤：

### 方案 A（推荐）：跳过清理，直接保存 ✅
1. **不要再运行清理cell** - 这会再次崩溃
2. **直接运行下面的"保存模型"cell**
3. 在Google Colab中，保存完成后会自动释放部分内存
4. 如果保存时仍然OOM：
   - 停止当前运行 (⏹️ Runtime → Interrupt execution)
   - 重启kernel (🔄 Runtime → Restart runtime)
   - 模型已保存到Google Drive，不会丢失

### 方案 B（如果仍然OOM）：直接跳到测试推理
1. 如果保存也报错，说明显存极度紧张
2. 使用下面的"加载已保存的模型"cell来重新加载
3. 在新的kernel中进行推理，无需清理步骤

### 原因说明
- 即使设置为 `None`，删除trainer/model的引用计数过程仍需临时内存
- 频繁的 `gc.collect()` 反而会增加内存压力
- 最安全的做法是：**让kernel在session结束时自动清理**

---

In [ ]:
print("\n" + "="*70)
print("💾 安全保存训练好的模型（支持OOM恢复）")
print("="*70)

import os
import json
import sys

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    # 第1步：保存PEFT LoRA权重
    print("\n1️⃣ 保存PEFT LoRA权重...")
    trainer.save_model(OUTPUT_DIR)
    print(f"   ✅ LoRA权重已保存到: {OUTPUT_DIR}")

except Exception as e:
    print(f"   ❌ 保存失败: {e}")
    print(f"\n💡 如果出现'Model not found'错误，说明trainer对象已被清理")
    print(f"   此时模型可能已经保存到checkpoint中")
    raise

try:
    # 第2步：保存tokenizer
    print("\n2️⃣ 保存tokenizer...")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/tokenizer")
    print(f"   ✅ Tokenizer已保存")

except Exception as e:
    print(f"   ⚠️  Tokenizer保存失败: {e}")

try:
    # 第3步：保存训练配置信息（轻量级操作）
    print("\n3️⃣ 保存训练信息...")
    training_info = {
        "model_name": MODEL_NAME,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
        "learning_rate": LEARNING_RATE,
        "warmup_steps": WARMUP_STEPS,
        "max_length": MAX_LENGTH,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "precision": "bfloat16",
        "optimizer": "adamw_torch",
        "data_level": "step_level",
        "training_date": "2026-03-05",
    }

    with open(f"{OUTPUT_DIR}/training_info.json", 'w', encoding='utf-8') as f:
        json.dump(training_info, f, indent=2, ensure_ascii=False)

    print(f"   ✅ 训练信息已保存")

except Exception as e:
    print(f"   ⚠️  配置保存失败: {e}")

try:
    # 第4步：保存简化版README（避免复杂字符串操作）
    print("\n4️⃣ 保存使用说明...")
    readme = """# GIS CodeLlama LoRA Model

## 快速开始
加载模型后，使用以下代码进行推理：

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "codellama/CodeLlama-7b-Instruct-hf",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "path/to/model/")
tokenizer = AutoTokenizer.from_pretrained("path/to/model/tokenizer/")

prompt = "Your instruction here"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0]))
```

更多详情见training_info.json。
"""

    with open(f"{OUTPUT_DIR}/README.md", 'w', encoding='utf-8') as f:
        f.write(readme)

    print(f"   ✅ README已保存")

except Exception as e:
    print(f"   ⚠️  README保存失败: {e}")

print("\n" + "="*70)
print("✅ 模型保存完成！")
print("="*70)
print(f"\n📁 模型已保存到：{OUTPUT_DIR}")
print("\n✨ 下一步：直接运行推理测试cell，无需运行内存清理cell")

## 🧪 步骤8：测试模型效果

## 🔄 从已保存的模型加载推理（推荐用于OOM情况）

**如果上面的训练和保存步骤出现OOM，使用这个cell在新kernel中加载模型进行推理！**

步骤：
1. 完成上面的"保存模型"cell（即使看起来有问题）
2. **重启kernel** (🔄 Runtime → Restart runtime)
3. 运行下面这个cell加载已保存的模型
4. 然后进行推理测试

---

In [ ]:
print("="*70)
print("🔄 从已保存的模型加载")
print("="*70)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_PATH = "/content/drive/MyDrive/gis-models/step-level-model"

print(f"\n📂 模型路径: {MODEL_PATH}")

# 检查模型是否存在
import os
if not os.path.exists(MODEL_PATH):
    print("❌ 模型路径不存在！")
    print("   请确保已运行'保存模型'cell，且模型已保存到Google Drive")
else:
    print("✅ 模型路径存在")

print("\n🤖 加载基础模型...")
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        "codellama/CodeLlama-7b-Instruct-hf",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    print("   ✅ 基础模型加载成功")
except Exception as e:
    print(f"   ❌ 加载失败: {e}")
    raise

print("\n📖 加载Tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(f"{MODEL_PATH}/tokenizer")
    # ⚠️ 修复：设置pad_token_id避免与eos_token冲突
    if tokenizer.pad_token is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    print("   ✅ Tokenizer加载成功")
except Exception as e:
    print(f"   ⚠️  Tokenizer加载失败: {e}")
    print("   使用默认tokenizer代替")
    tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-Instruct-hf")
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("\n🔧 加载LoRA权重...")
try:
    model = PeftModel.from_pretrained(base_model, MODEL_PATH)
    print("   ✅ LoRA权重加载成功")
    model.print_trainable_parameters()
except Exception as e:
    print(f"   ❌ LoRA加载失败: {e}")
    print("   使用基础模型进行推理（无LoRA微调）")
    model = base_model

# 切换到评估模式
model.eval()

print("\n✅ 模型加载完成！")
print("\n💾 显存状态:")
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"   已用: {allocated:.2f} GB / {total:.2f} GB")

print("\n" + "="*70)
print("现在可以运行下面的推理测试cell了！")
print("="*70)

In [ ]:
import os
import json

print("="*70)
print("🔍 深度诊断：检查checkpoint权重文件")
print("="*70)

checkpoint_dir = "/content/drive/MyDrive/gis-models/step-level-model"

print(f"\n📂 检查目录: {checkpoint_dir}\n")

# 检查每个checkpoint的详细内容
for ckpt_name in sorted(os.listdir(checkpoint_dir)):
    ckpt_path = os.path.join(checkpoint_dir, ckpt_name)
    
    if not os.path.isdir(ckpt_path):
        continue
    
    print(f"📁 {ckpt_name}:")
    print("-" * 50)
    
    files = os.listdir(ckpt_path)
    
    # 关键文件检查
    critical_files = {
        'adapter_config.json': 'LoRA配置',
        'adapter_model.bin': '⚠️ LoRA权重（最关键！）',
        'training_args.bin': '训练参数',
        'pytorch_model.bin': '基础模型权重',
        'config.json': '模型配置',
    }
    
    for fname, description in critical_files.items():
        fpath = os.path.join(ckpt_path, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"  ✅ {fname:30s} | {size_mb:8.1f} MB | {description}")
        else:
            print(f"  ❌ {fname:30s} | {'缺失':8s} | {description}")
    
    # 列出所有文件
    print(f"\n  📋 完整文件列表:")
    for fname in sorted(files):
        fpath = os.path.join(ckpt_path, fname)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"     - {fname:40s} {size_mb:8.1f} MB")
    
    print()

# 问题分析
print("="*70)
print("🔴 问题分析")
print("="*70)

# 检查adapter_model.bin是否在任何checkpoint中
has_adapter_model = False
for ckpt_name in os.listdir(checkpoint_dir):
    ckpt_path = os.path.join(checkpoint_dir, ckpt_name)
    if os.path.isdir(ckpt_path):
        if os.path.exists(os.path.join(ckpt_path, 'adapter_model.bin')):
            has_adapter_model = True
            print(f"\n✅ 找到adapter_model.bin: {ckpt_name}")

if not has_adapter_model:
    print("\n❌ 严重问题：所有checkpoint都缺少 adapter_model.bin")
    print("\n这意味着：")
    print("  1. LoRA权重没有被正确保存")
    print("  2. trainer.save_model() 可能失败了")
    print("  3. 或者训练中根本没有应用LoRA")
    print("\n原因可能：")
    print("  • save_strategy='no' 禁用了中间checkpoint保存")
    print("  • trainer.train() 后没有调用 trainer.save_model()")
    print("  • 模型保存时出现OOM错误")
    print("\n⚠️ 需要重新训练！")
else:
    print("\n✅ 找到了adapter_model.bin，模型应该可以加载")

# 检查是否有其他模型保存位置
print("\n📍 检查其他可能的保存位置...")
print("-" * 70)

check_paths = [
    "/content/drive/MyDrive/gis-models/step-level-model",
    "/content/drive/MyDrive/gis-models",
    "/content/step-level-model",
]

for check_path in check_paths:
    if os.path.exists(check_path):
        print(f"✅ 存在: {check_path}")
        # 列出第一层文件/目录
        try:
            items = os.listdir(check_path)
            for item in items[:10]:  # 只显示前10个
                item_path = os.path.join(check_path, item)
                if os.path.isdir(item_path):
                    print(f"   📁 {item}/")
                else:
                    size_mb = os.path.getsize(item_path) / (1024 * 1024)
                    print(f"   📄 {item} ({size_mb:.1f}MB)")
            if len(items) > 10:
                print(f"   ... 及其他 {len(items)-10} 项")
        except:
            pass
    else:
        print(f"❌ 不存在: {check_path}")

print("\n" + "="*70)


In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("="*70)
print("🎉 好消息：权重已保存！（safetensors格式）")
print("="*70)

print("\n✅ 问题诊断：")
print("   • adapter_model.safetensors 存在（305.1 MB）")
print("   • 这就是你的LoRA权重文件！")
print("   • 现代PEFT使用safetensors格式（更安全）")
print("   • 之前的诊断代码查找.bin格式，所以没找到")

print("\n🔧 现在加载checkpoint-50的权重...\n")

checkpoint_path = "/content/drive/MyDrive/gis-models/step-level-model/checkpoint-50"

print(f"1️⃣ 加载基础模型...")
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        "codellama/CodeLlama-7b-Instruct-hf",
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    print("   ✅ 成功\n")
except Exception as e:
    print(f"   ❌ 失败: {e}\n")
    raise

print(f"2️⃣ 加载LoRA权重（safetensors格式）...")
try:
    # PeftModel.from_pretrained 会自动检测safetensors
    model = PeftModel.from_pretrained(
        base_model, 
        checkpoint_path,
        is_trainable=False  # 推理模式
    )
    print("   ✅ 成功\n")
    model.print_trainable_parameters()
except Exception as e:
    print(f"   ❌ 失败: {e}")
    print(f"\n   尝试手动加载safetensors...")
    try:
        from safetensors.torch import load_file
        
        # 手动加载
        safetensors_path = os.path.join(checkpoint_path, "adapter_model.safetensors")
        state_dict = load_file(safetensors_path)
        print(f"   ✅ 加载了 {len(state_dict)} 个张量")
        
        model = PeftModel.from_pretrained(base_model, checkpoint_path)
        print("   ✅ LoRA模型创建成功")
    except ImportError:
        print("   需要安装safetensors")
        os.system("pip install -q safetensors")
        model = PeftModel.from_pretrained(base_model, checkpoint_path)
        print("   ✅ LoRA模型创建成功")

print(f"\n3️⃣ 加载tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint_path,  # checkpoint中有tokenizer
    )
    print("   ✅ 成功\n")
except:
    print("   ⚠️ checkpoint中的tokenizer加载失败，使用默认...")
    tokenizer = AutoTokenizer.from_pretrained(
        "codellama/CodeLlama-7b-Instruct-hf"
    )
    print("   ✅ 使用默认tokenizer\n")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print("="*70)
print("✅ 模型加载完成！")
print("="*70)
print("\n📊 模型信息：")
print(f"   模型类型: {type(model).__name__}")
print(f"   设备: {next(model.parameters()).device}")
print(f"   精度: {next(model.parameters()).dtype}")
print(f"   Tokenizer词表大小: {len(tokenizer)}")

print("\n🎯 现在可以直接推理了！")
print("   运行下一个cell进行推理测试")


In [ ]:
import json
import torch
from tqdm import tqdm

print("="*70)
print("🧪 完整推理测试（验证模型能否生成JSON）")
print("="*70)

print(f"\n📊 测试配置：")
print(f"   验证集大小: {len(val_data)} 样本")
print(f"   本次测试: 前10个样本")
print(f"   最大生成长度: 200 tokens\n")

# ============================================================
# 推理函数
# ============================================================

def infer_sample(instruction, max_tokens=200):
    """对单个样本进行推理"""
    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding="max_length",
        return_attention_mask=True
    )
    
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)
    
    # 生成
    with torch.no_grad():
        torch.cuda.empty_cache()
        
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams=1,
        )
    
    # 解码
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取JSON部分
    if "Step Data JSON:" in generated_text:
        json_part = generated_text.split("Step Data JSON:")[-1].strip()
    else:
        json_part = generated_text
    
    return json_part

# ============================================================
# 运行推理
# ============================================================

results = {
    'total': 0,
    'success': 0,
    'has_json': 0,
    'valid_json': 0,
    'samples': []
}

print("⏳ 开始推理...\n")

for idx in tqdm(range(min(10, len(val_data))), desc="进度"):
    try:
        sample = val_data[idx]
        instruction = sample['instruction']
        
        # 推理
        json_output = infer_sample(instruction, max_tokens=150)
        
        results['success'] += 1
        results['total'] += 1
        
        # 尝试解析JSON
        valid_json = False
        try:
            json_obj = json.loads(json_output)
            results['valid_json'] += 1
            valid_json = True
            status = "✅ JSON有效"
        except json.JSONDecodeError:
            if json_output.strip().startswith('{'):
                results['has_json'] += 1
                status = "⚠️ JSON格式错"
            else:
                status = "❌ 无JSON"
        
        results['samples'].append({
            'index': idx,
            'instruction': instruction[:50],
            'output_preview': json_output[:80],
            'is_valid': valid_json,
            'status': status
        })
        
    except Exception as e:
        results['total'] += 1
        results['samples'].append({
            'index': idx,
            'instruction': val_data[idx]['instruction'][:50],
            'output_preview': f"Error: {str(e)[:50]}",
            'is_valid': False,
            'status': "❌ 错误"
        })

# ============================================================
# 结果统计
# ============================================================

print("\n" + "="*70)
print("📊 推理结果")
print("="*70)

print(f"\n总样本：{results['total']}")
print(f"推理成功：{results['success']}/{results['total']}")
print(f"包含JSON：{results['has_json']}/{results['total']}")
print(f"✅ JSON有效：{results['valid_json']}/{results['total']}")

success_rate = (results['valid_json'] / results['total'] * 100) if results['total'] > 0 else 0
print(f"\n🎯 成功率：{success_rate:.1f}%")

# ============================================================
# 样本详情
# ============================================================

print(f"\n📝 推理样本详情：")
print("-" * 70)

for sample in results['samples']:
    print(f"\n样本 #{sample['index']} {sample['status']}")
    print(f"  指令: {sample['instruction']}...")
    print(f"  输出: {sample['output_preview']}...")

# ============================================================
# 结论
# ============================================================

print("\n" + "="*70)
print("📈 结论")
print("="*70)

if results['valid_json'] > 0:
    print(f"\n✅ 🎉 好消息！模型训练成功！")
    print(f"   • 能够生成有效JSON")
    print(f"   • JSON有效率：{success_rate:.1f}%")
    print(f"\n💡 下一步建议：")
    print(f"   • 如果想提高准确率，增加训练样本到1000+")
    print(f"   • 或者增加训练轮次到5-10")
    print(f"   • 当前模型可以用于演示和原型开发")
else:
    print(f"\n⚠️ 模型无法生成有效JSON")
    print(f"\n可能的原因：")
    print(f"   1. 训练数据太少（只有~100样本）")
    print(f"   2. 训练轮次不足（只有3轮）")
    print(f"   3. 提示词格式与训练不一致")
    print(f"\n建议：")
    print(f"   • 增加训练数据到1000+样本")
    print(f"   • 重新训练10-20轮")
    print(f"   • 调整LoRA参数（r=64, alpha=32）")

print("\n" + "="*70)


In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("="*70)
print("🚨 GPU损坏 → 切换到CPU模式")
print("="*70)

print("\n⚠️ CUDA device-side assert 错误表示GPU状态已破坏")
print("   解决方案：使用CPU推理（慢但稳定）\n")

checkpoint_path = "/content/drive/MyDrive/gis-models/step-level-model/checkpoint-50"

print(f"1️⃣ 加载基础模型（CPU模式）...")
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        "codellama/CodeLlama-7b-Instruct-hf",
        torch_dtype=torch.float32,  # CPU上用float32更稳定
        device_map="cpu",  # ⚠️ 强制CPU
    )
    print("   ✅ 成功\n")
except Exception as e:
    print(f"   ❌ 失败: {e}\n")
    raise

print(f"2️⃣ 加载LoRA权重...")
try:
    model = PeftModel.from_pretrained(
        base_model, 
        checkpoint_path,
        is_trainable=False
    )
    print("   ✅ 成功\n")
except Exception as e:
    print(f"   ❌ 失败: {e}\n")
    raise

print(f"3️⃣ 加载tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("   ✅ 成功\n")

model.eval()

print("="*70)
print("✅ 模型加载完成（CPU模式）")
print("="*70)
print("\n⚠️ 注意：CPU推理很慢（1样本≈30秒）")
print("   但GPU损坏时这是唯一的选择")
print("\n💡 长期解决方案：重启kernel后用GPU推理")


In [ ]:
import json
import torch
from tqdm import tqdm

print("="*70)
print("🧪 推理测试（CPU模式）")
print("="*70)

print(f"\n⏱️ 预计时间：每个样本约30秒")
print(f"   总测试时间：3个样本 ≈ 90秒（1.5分钟）\n")

# ============================================================
# 推理函数（CPU安全版）
# ============================================================

def infer_sample_cpu(instruction, max_tokens=100):
    """CPU安全的推理函数"""
    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""
    
    # Tokenize - 不指定device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=False,  # CPU避免padding
    )
    
    # 生成 - 最小化参数
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams=1,
            # 不使用任何GPU相关参数
        )
    
    # 解码
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取JSON
    if "Step Data JSON:" in generated_text:
        json_part = generated_text.split("Step Data JSON:")[-1].strip()
    else:
        json_part = generated_text
    
    return json_part

# ============================================================
# 运行快速测试（3个样本）
# ============================================================

results = {
    'total': 0,
    'success': 0,
    'valid_json': 0,
    'samples': []
}

print("⏳ 开始推理（只测试3个样本以节省时间）...\n")

for idx in tqdm(range(min(3, len(val_data))), desc="进度"):
    try:
        sample = val_data[idx]
        instruction = sample['instruction']
        
        # 推理
        json_output = infer_sample_cpu(instruction, max_tokens=80)
        
        results['success'] += 1
        results['total'] += 1
        
        # 尝试解析JSON
        valid = False
        try:
            json_obj = json.loads(json_output)
            results['valid_json'] += 1
            valid = True
            status = "✅ JSON有效"
        except:
            status = "❌ JSON无效" if json_output.startswith('{') else "❌ 无JSON"
        
        results['samples'].append({
            'idx': idx,
            'instr': instruction[:40],
            'output': json_output[:60],
            'valid': valid,
            'status': status
        })
        
    except Exception as e:
        results['total'] += 1
        results['samples'].append({
            'idx': idx,
            'instr': val_data[idx]['instruction'][:40],
            'output': f"Error: {str(e)[:40]}",
            'valid': False,
            'status': "❌ 错误"
        })

# ============================================================
# 结果
# ============================================================

print("\n" + "="*70)
print("📊 推理结果")
print("="*70)

print(f"\n总样本：{results['total']}")
print(f"推理成功：{results['success']}/{results['total']}")
print(f"✅ JSON有效：{results['valid_json']}/{results['total']}")

success_rate = (results['valid_json'] / results['total'] * 100) if results['total'] > 0 else 0
print(f"\n🎯 JSON有效率：{success_rate:.1f}%")

print(f"\n📝 样本详情：")
for s in results['samples']:
    print(f"\n样本 #{s['idx']} {s['status']}")
    print(f"  指令: {s['instr']}...")
    print(f"  输出: {s['output']}...")

# ============================================================
# 结论
# ============================================================

print("\n" + "="*70)
print("📈 结论")
print("="*70)

if results['valid_json'] > 0:
    print(f"\n✅ 模型能生成有效JSON！")
    print(f"   有效率：{success_rate:.1f}%")
    print(f"\n这表明你的训练成功了！")
else:
    print(f"\n⚠️ 从这3个样本看，模型难以生成有效JSON")
    print(f"   可能需要增加训练数据或轮次")

print("\n💡 下一步：")
print(f"   1. 测试更多样本（运行带10个样本的完整版推理）")
print(f"   2. 或重启kernel，用GPU推理会快10倍")
print(f"   3. 如果有效率<50%，重新训练增加数据到1000+")

print("\n" + "="*70)


In [ ]:
print("🧪 模型推理测试\n")
print("="*70)
print("使用训练好的模型进行推理")
print("="*70)

# 确保模型处于评估模式
model.eval()

# 定义推理函数
def test_model(instruction, context="", max_tokens=200):
    """
    测试模型推理

    Args:
        instruction: 自然语言指令
        context: 上下文信息（可选）
        max_tokens: 最大生成token数

    Returns:
        生成的文本
    """
    # 构建prompt
    if context:
        prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Context: {context}
Instruction: {instruction}

Step Data JSON:"""
    else:
        prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""

    # Tokenize - ⚠️ 修复：明确返回attention_mask
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=256,
        padding=True,
        return_attention_mask=True
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # 生成 - ⚠️ 修复：传递attention_mask，disable sampling
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=False,  # ✅ 改为False避免CUDA multinomial错误
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 解码
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 提取JSON部分
    if "Step Data JSON:" in generated_text:
        response = generated_text.split("Step Data JSON:")[-1].strip()
    else:
        response = generated_text

    return response

# 定义测试用例
test_cases = [
    {
        "name": "测试1: 打开电缆对象",
        "instruction": "Open E MS Kabel",
        "context": "Database: :elektra | Application: PowerGrid",
    },
    {
        "name": "测试2: 创建新站点",
        "instruction": "Create new station building",
        "context": "Location: Amsterdam | Type: MS Station",
    },
    {
        "name": "测试3: 设置接地变压器",
        "instruction": "Set MS HS aardingstrafo asset properties",
        "context": "Database: ND | Operation Type: Create",
    },
    {
        "name": "测试4: 添加高压连接",
        "instruction": "Setup HS cable connection between two nodes",
        "context": "Voltage Level: 110kV | Cable Type: Underground",
    },
    {
        "name": "测试5: 配置低压安装",
        "instruction": "Configure LV installation with protection devices",
        "context": "Building Type: Industrial | Safety Level: High",
    },
]

# 执行测试
print("\n")
for i, test_case in enumerate(test_cases, 1):
    print("="*70)
    print(f"🔮 {test_case['name']}")
    print("="*70)
    print(f"\n📝 输入信息:")
    print(f"   指令: {test_case['instruction']}")
    print(f"   上下文: {test_case['context']}")

    print(f"\n⏳ 生成中...\n")

    result = test_model(
        instruction=test_case['instruction'],
        context=test_case['context'],
        max_tokens=200
    )

    print(f"📤 输出结果:")
    print(f"{result}")

    # 尝试解析JSON
    try:
        import json
        json_obj = json.loads(result)
        print(f"\n✅ JSON有效性: 有效")
        print(f"📊 关键字段:")
        if 'step_index' in json_obj:
            print(f"   - step_index: {json_obj['step_index']}")
        if 'object' in json_obj:
            print(f"   - object: {json_obj['object']}")
        if 'method' in json_obj:
            print(f"   - method: {json_obj['method']}")
        if 'database' in json_obj:
            print(f"   - database: {json_obj['database']}")
    except json.JSONDecodeError:
        print(f"\n⚠️  JSON有效性: 无效 (未能解析)")

    print("\n")

print("="*70)
print("🎉 测试完成！")
print("="*70)

## 🎯 步骤9：交互式推理（自定义输入）

In [ ]:
print("🎯 交互式模型测试\n")
print("使用下面的代码自定义GIS指令来测试模型")
print("="*70)

# 自定义输入 - 修改这些变量来测试不同的指令
your_instruction = "Open E MS Kabel"  # 修改这里：输入你的GIS指令
your_context = "Database: :elektra | Application: PowerGrid"  # 修改这里：输入背景信息

print(f"\n📝 你的输入:")
print(f"   指令: {your_instruction}")
print(f"   上下文: {your_context}")

print(f"\n⏳ 生成中...\n")

# 生成结果
result = test_model(
    instruction=your_instruction,
    context=your_context,
    max_tokens=300
)

print("📤 模型输出:")
print("-" * 70)
print(result)
print("-" * 70)

# 尝试解析和美化JSON结果
try:
    import json
    json_result = json.loads(result)

    print("\n✅ JSON解析成功！")
    print("\n📊 结构化数据:")
    print(json.dumps(json_result, indent=2, ensure_ascii=False))

    # 提取关键信息
    print("\n🔍 关键信息提取:")
    for key, value in json_result.items():
        if not isinstance(value, (dict, list)):
            print(f"   • {key}: {value}")
        elif isinstance(value, list):
            print(f"   • {key}: [{len(value)} 项]")

except json.JSONDecodeError as e:
    print(f"\n⚠️  JSON解析失败")
    print(f"   错误: {str(e)[:100]}")
    print(f"\n💡 提示:")
    print(f"   • 模型可能需要更多训练来生成有效的JSON")
    print(f"   • 尝试更改指令或提供更详细的上下文")
    print(f"   • 增加BATCH_SIZE或训练轮数可能会改善结果")

print("\n" + "="*70)
print("💡 提示: 修改上面代码中的 your_instruction 和 your_context 来测试其他指令")

## 🚀 GPU推理版本（重启kernel后使用）

**重要：在Colab中必须先重启kernel**
- 菜单：`Runtime` → `Restart runtime`
- 等待30秒完全启动
- 然后运行下面的cells

这样GPU会恢复正常，可以进行快速高效的推理。

---


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("="*70)
print("🚀 加载checkpoint-50（GPU模式）")
print("="*70)

checkpoint_path = "/content/drive/MyDrive/gis-models/step-level-model/checkpoint-50"

print("\n1️⃣ 加载基础模型...\n")
base_model = AutoModelForCausalLM.from_pretrained(
    "codellama/CodeLlama-7b-Instruct-hf",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
print("✅ 成功")

print("\n2️⃣ 加载LoRA权重...\n")
model = PeftModel.from_pretrained(
    base_model, 
    checkpoint_path,
    is_trainable=False
)
print("✅ 成功")
model.print_trainable_parameters()

print("\n3️⃣ 加载tokenizer...\n")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ 成功")

model.eval()

print("\n" + "="*70)
print("✅ 模型已加载（GPU模式）")
print("="*70)
print(f"   设备: {next(model.parameters()).device}")
print(f"   精度: {next(model.parameters()).dtype}\n")


In [ ]:
import json
import torch
from tqdm import tqdm

print("="*70)
print("🧪 推理测试（GPU模式 - 快速）")
print("="*70)

print(f"\n📊 测试配置：")
print(f"   验证集大小: {len(val_data)} 样本")
print(f"   本次测试: 前10个样本")
print(f"   预计时间: 30-60秒\n")

# ============================================================
# 推理函数
# ============================================================

def infer_sample(instruction, max_tokens=150):
    """GPU推理函数"""
    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding="max_length",
        return_attention_mask=True
    )
    
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)
    
    # 生成
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams=1,
        )
    
    # 解码
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取JSON
    if "Step Data JSON:" in generated_text:
        json_part = generated_text.split("Step Data JSON:")[-1].strip()
    else:
        json_part = generated_text
    
    return json_part

# ============================================================
# 运行推理
# ============================================================

results = {
    'total': 0,
    'success': 0,
    'valid_json': 0,
    'samples': []
}

print("⏳ 开始推理...\n")

for idx in tqdm(range(min(10, len(val_data))), desc="进度"):
    try:
        sample = val_data[idx]
        instruction = sample['instruction']
        
        # 推理
        json_output = infer_sample(instruction, max_tokens=120)
        
        results['success'] += 1
        results['total'] += 1
        
        # 尝试解析JSON
        valid_json = False
        try:
            json_obj = json.loads(json_output)
            results['valid_json'] += 1
            valid_json = True
            status = "✅"
        except json.JSONDecodeError:
            status = "⚠️" if json_output.strip().startswith('{') else "❌"
        
        results['samples'].append({
            'index': idx,
            'instruction': instruction[:50],
            'output': json_output[:70],
            'valid': valid_json,
            'status': status
        })
        
    except Exception as e:
        results['total'] += 1
        results['samples'].append({
            'index': idx,
            'instruction': val_data[idx]['instruction'][:50],
            'output': f"Error",
            'valid': False,
            'status': "❌"
        })

# ============================================================
# 结果统计
# ============================================================

print("\n" + "="*70)
print("📊 推理结果")
print("="*70)

print(f"\n总样本：{results['total']}")
print(f"推理成功：{results['success']}/{results['total']}")
print(f"✅ JSON有效：{results['valid_json']}/{results['total']}")

success_rate = (results['valid_json'] / results['total'] * 100) if results['total'] > 0 else 0
print(f"\n🎯 JSON有效率：{success_rate:.1f}%")

# ============================================================
# 样本详情
# ============================================================

print(f"\n📝 推理样本详情：")
print("-" * 70)

for sample in results['samples'][:5]:  # 只显示前5个
    print(f"\n样本 #{sample['index']} {sample['status']}")
    print(f"  指令: {sample['instruction']}...")
    print(f"  输出: {sample['output']}...")

# ============================================================
# 结论
# ============================================================

print("\n" + "="*70)
print("📈 结论")
print("="*70)

if results['valid_json'] > 0:
    print(f"\n✅ 🎉 模型能生成有效JSON！")
    print(f"   有效率：{success_rate:.1f}%")
    print(f"\n🏆 模型训练成功！")
else:
    print(f"\n⚠️ 模型难以生成有效JSON")
    print(f"   建议：增加训练样本到1000+或增加训练轮次")

print("\n" + "="*70)
